# LeetCode 504 — Base 7 bằng Python

**Link đề:** https://leetcode.com/problems/base-7/
**Độ khó:** Easy · **Chủ đề:** Math, String

```python
class Solution:
    def convertToBase7(self, num: int) -> str:
        ...
```

## Mục tiêu học
1. Hiểu **hệ cơ số** (base): mỗi vị trí có "trọng số" là lũy thừa của cơ số.
2. Nắm chắc `%` (lấy dư) và `//` (chia lấy nguyên), `divmod`, và cách chúng hoạt động với **số âm** trong Python.
3. Thuật toán **chia liên tiếp lấy dư** để đổi từ hệ 10 sang hệ bất kỳ; xử lý `0` và số âm.
4. So sánh 4 cách: list chữ số + đảo, ghép chuỗi về bên trái, đệ quy, dựng "số trông như hệ 7".
5. Trace, chứng minh, test với `int(s, 7)`, và tổng quát hoá sang hệ 2..36.

## 1. Đề bài

Cho số nguyên `num`, trả về **chuỗi** biểu diễn `num` trong **hệ cơ số 7**.

| num | Output | Kiểm tra |
|---:|---|---|
| `100` | `"202"` | `2·49 + 0·7 + 2·1 = 100` |
| `-7` | `"-10"` | `-(1·7 + 0) = -7` |

**Constraints:** `-10^7 <= num <= 10^7`.

Chú ý: đầu ra là **chuỗi** (`str`), có dấu `-` nếu âm, và `0` → `"0"`.

## 2. Nền tảng: hệ cơ số là gì?

Trong hệ 10, `352 = 3·10² + 5·10¹ + 2·10⁰`. Mỗi chữ số thuộc `0..9`.

Trong hệ 7, chữ số thuộc `0..6`, trọng số là `7⁰=1, 7¹=7, 7²=49, 7³=343, ...`

`"202"` (hệ 7) `= 2·49 + 0·7 + 2·1 = 100` (hệ 10).

Ta kiểm chứng bằng code — và dùng luôn hàm có sẵn `int(s, 7)` của Python để **đọc** chuỗi hệ 7 (rất tiện để test, nhưng đề muốn ta tự **viết** chiều ngược lại).

In [1]:
print(2 * 7**2 + 0 * 7**1 + 2 * 7**0)   # 100
print(int("202", 7))                    # 100 — Python đọc chuỗi hệ 7
print(int("-10", 7))                    # -7
print([7**k for k in range(6)])         # các trọng số

100
100
-7
[1, 7, 49, 343, 2401, 16807]


### 2.1. `%` và `//` — lấy chữ số cuối, bỏ chữ số cuối

Giống hệ 10 với `% 10`, `// 10`, trong hệ 7:
- `n % 7` = **chữ số cuối** (hàng đơn vị) của `n` khi viết ở hệ 7.
- `n // 7` = `n` sau khi **bỏ chữ số cuối** đó.

Vì sao? `n = (n // 7)·7 + (n % 7)` và `0 ≤ n % 7 < 7`. Phần `(n // 7)·7` đều chia hết cho 7, nên mọi chữ số khác chỉ nằm ở các hàng cao hơn.

In [2]:
n = 100
print(n % 7, n // 7)      # 2 14   -> chữ số cuối 2, còn lại 14
print(14 % 7, 14 // 7)    # 0 2
print(2 % 7, 2 // 7)      # 2 0    -> dừng vì còn lại 0
print("đọc ngược các số dư: 2, 0, 2 -> '202'")

2 14
0 2
2 0
đọc ngược các số dư: 2, 0, 2 -> '202'


### 2.2. `divmod(a, b)` — lấy cả thương và dư một lần

In [3]:
q, r = divmod(100, 7)
print(q, r)          # 14 2
print(divmod(14, 7)) # (2, 0)

14 2
(2, 0)


### 2.3. ⚠️ `%` và `//` với số âm trong Python

Python làm tròn `//` **xuống** (về âm vô cùng), nên:
- `-7 // 7 = -1`, `-7 % 7 = 0` (ổn)
- `-100 // 7 = -15`, `-100 % 7 = 5` (vì `-15·7 + 5 = -100`) → **không** phải chữ số mong muốn!

Bài học: **tách dấu ra trước**, làm việc với `abs(num)`, cuối cùng mới gắn `-`.

In [4]:
print(-100 // 7, -100 % 7)     # -15 5   (khác với C/Java: -14, -2)
print(abs(-100) // 7, abs(-100) % 7)

-15 5
14 2


### 2.4. Ghép chuỗi: `str(d)`, `"".join(list)`, đảo `[::-1]`

- `str(2)` → `"2"`.
- `"".join(["2", "0", "2"])` → `"202"` (nối các chuỗi trong list, ngăn cách bởi chuỗi rỗng).
- `lst[::-1]` hoặc `reversed(lst)` để đảo thứ tự.

In [5]:
digits = [2, 0, 2]
print("".join(str(d) for d in digits))
print([2, 0, 3][::-1], list(reversed([2, 0, 3])))
print("".join(map(str, [6, 5, 4])))   # map(str, ...) áp dụng str cho từng phần tử

202
[3, 0, 2] [3, 0, 2]
654


## 3. Cách trực giác — trừ dần các lũy thừa lớn nhất

Người mới thường nghĩ giống "đổi tiền": tìm lũy thừa lớn nhất `7^k ≤ n`, xem chứa được bao nhiêu lần, trừ đi, rồi chuyển xuống `7^(k-1)`...

`100`: `7² = 49` chứa 2 lần (98), dư 2 → `7¹` chứa 0 lần → `7⁰` chứa 2 lần → `"202"`.

In [6]:
def to_base7_greedy(num):
    if num == 0:
        return "0"
    sign = "-" if num < 0 else ""
    n = abs(num)
    power = 1
    while power * 7 <= n:          # tìm lũy thừa lớn nhất <= n
        power *= 7
    digits = []
    while power >= 1:
        digits.append(str(n // power))
        n %= power
        power //= 7
    return sign + "".join(digits)

for x in [100, -7, 0, 6, 49, 48]:
    print(x, "->", to_base7_greedy(x))

100 -> 202
-7 -> -10
0 -> 0
6 -> 6
49 -> 100
48 -> 66


### Nút thắt của cách trực giác

Đúng, nhưng phải **đi hai lượt**: một lượt tìm lũy thừa lớn nhất, một lượt chia. Còn dễ sai ở biên (`power * 7 <= n` hay `<`?).

Cách gọn hơn: **lấy chữ số từ phải sang trái** bằng `% 7` và `// 7` — không cần biết trước số có bao nhiêu chữ số.

## 4. Insight — chia liên tiếp lấy dư

```text
n = 100: 100 % 7 = 2, n = 14
n = 14 :  14 % 7 = 0, n = 2
n = 2  :   2 % 7 = 2, n = 0  -> dừng
Các số dư theo thứ tự thu được: 2, 0, 2  (hàng đơn vị trước)
Đảo lại: "202"
```
Các chữ số ra theo thứ tự **từ hàng thấp lên hàng cao**, nên cuối cùng phải **đảo**.

## 5. Lời giải chính — mổ xẻ từng đoạn

1. **Trường hợp `0`**: vòng `while n > 0` không chạy lần nào → chuỗi rỗng. Phải trả `"0"` riêng.
2. **Tách dấu**: `negative = num < 0`, `n = abs(num)`.
3. **Vòng lặp**: `digits.append(str(n % 7))`, `n //= 7`.
4. **Đảo + gắn dấu**: `("-" if negative else "") + "".join(reversed(digits))`.

In [7]:
def to_base7_divide(num):
    if num == 0:
        return "0"
    negative = num < 0
    n = abs(num)
    digits = []
    while n > 0:
        digits.append(str(n % 7))
        n //= 7
    result = "".join(reversed(digits))
    return "-" + result if negative else result

for x in [100, -7, 0, 1, -1, 6, 7, 10**7, -10**7]:
    print(f"{x:>9} -> {to_base7_divide(x)}")

      100 -> 202
       -7 -> -10
        0 -> 0
        1 -> 1
       -1 -> -1
        6 -> 6
        7 -> 10
 10000000 -> 150666343
-10000000 -> -150666343


## 6. TRACE từng bước

`num = -2024`:

| bước | n trước | n % 7 | n // 7 | digits sau |
|---:|---:|---:|---:|---|
| 1 | 2024 | 1 | 289 | `['1']` |
| 2 | 289 | 2 | 41 | `['1','2']` |
| 3 | 41 | 6 | 5 | `['1','2','6']` |
| 4 | 5 | 5 | 0 | `['1','2','6','5']` |

Đảo → `"5621"`, gắn dấu → `"-5621"`. Kiểm tra: `5·343 + 6·49 + 2·7 + 1 = 1715 + 294 + 14 + 1 = 2024`. ✔

In [8]:
def trace_base7(num):
    print(f"num = {num}")
    if num == 0:
        print("num = 0 -> trả '0'")
        return "0"
    negative = num < 0
    n = abs(num)
    digits = []
    step = 1
    while n > 0:
        r = n % 7
        print(f"bước {step}: n={n:>5}  n%7={r}  n//7={n // 7:>4}  digits={digits + [str(r)]}")
        digits.append(str(r))
        n //= 7
        step += 1
    result = ("-" if negative else "") + "".join(reversed(digits))
    print("đảo + dấu ->", repr(result), "| kiểm tra int(.,7) =", int(result, 7))
    return result

trace_base7(-2024)
print()
trace_base7(100)

num = -2024
bước 1: n= 2024  n%7=1  n//7= 289  digits=['1']
bước 2: n=  289  n%7=2  n//7=  41  digits=['1', '2']
bước 3: n=   41  n%7=6  n//7=   5  digits=['1', '2', '6']
bước 4: n=    5  n%7=5  n//7=   0  digits=['1', '2', '6', '5']
đảo + dấu -> '-5621' | kiểm tra int(.,7) = -2024

num = 100
bước 1: n=  100  n%7=2  n//7=  14  digits=['2']
bước 2: n=   14  n%7=0  n//7=   2  digits=['2', '0']
bước 3: n=    2  n%7=2  n//7=   0  digits=['2', '0', '2']
đảo + dấu -> '202' | kiểm tra int(.,7) = 100


'202'

## 7. Các cách khác

### 7.1. Ghép chuỗi về bên trái (không cần đảo)
Mỗi chữ số mới là chữ số **cao hơn**, nên đặt nó **trước** chuỗi hiện tại: `s = str(n % 7) + s`.
Ngắn gọn; nhưng mỗi lần ghép tạo chuỗi mới (chi phí `O(độ dài)`), số chữ số nhỏ nên không đáng kể.

In [9]:
def to_base7_prepend(num):
    if num == 0:
        return "0"
    n, s = abs(num), ""
    while n:
        n, r = divmod(n, 7)
        s = str(r) + s
    return ("-" if num < 0 else "") + s

print(to_base7_prepend(100), to_base7_prepend(-7), to_base7_prepend(0))

202 -10 0


### 7.2. Đệ quy
Biểu diễn của `n` = biểu diễn của `n // 7` **nối** chữ số `n % 7`. Điểm dừng: `n < 7` → một chữ số.

In [10]:
def to_base7_recursive(num):
    if num < 0:
        return "-" + to_base7_recursive(-num)
    if num < 7:
        return str(num)
    return to_base7_recursive(num // 7) + str(num % 7)

print(to_base7_recursive(100), to_base7_recursive(-7), to_base7_recursive(0))

202 -10 0


Cách đệ quy tự xử lý `0` (vì `0 < 7` → `"0"`) và số âm — rất gọn. Độ sâu đệ quy = số chữ số hệ 7 (≤ 10 với constraints).

### 7.3. Dựng một **số nguyên** "trông giống" hệ 7
Chữ số thứ `k` (từ phải, đếm từ 0) nhân với `10^k` rồi cộng dồn: `100 → 2·1 + 0·10 + 2·100 = 202`. Cuối cùng `str(...)`. Mẹo này hay gặp trong lời giải ngắn; dấu được xử lý nhờ `str` của số âm.

In [11]:
def to_base7_numeric(num):
    n, place, value = abs(num), 1, 0
    while n:
        value += (n % 7) * place
        place *= 10
        n //= 7
    return str(-value if num < 0 else value)

print(to_base7_numeric(100), to_base7_numeric(-7), to_base7_numeric(0))

202 -10 0


## 8. Code nộp LeetCode

### Bản chia liên tiếp (khuyên dùng)

In [12]:
class Solution:
    def convertToBase7(self, num: int) -> str:
        if num == 0:
            return "0"
        negative = num < 0
        n = abs(num)
        digits = []
        while n > 0:
            digits.append(str(n % 7))
            n //= 7
        result = "".join(reversed(digits))
        return "-" + result if negative else result

s = Solution()
print(s.convertToBase7(100), s.convertToBase7(-7))   # 202 -10

202 -10


### Bản đệ quy (ngắn)

In [13]:
class SolutionRecursive:
    def convertToBase7(self, num: int) -> str:
        if num < 0:
            return "-" + self.convertToBase7(-num)
        if num < 7:
            return str(num)
        return self.convertToBase7(num // 7) + str(num % 7)

print(SolutionRecursive().convertToBase7(100), SolutionRecursive().convertToBase7(-7))

202 -10


(Khi nộp, đổi tên lớp thành `Solution`.)

## 9. Vì sao đúng?

Gọi `n = abs(num) > 0`. Viết `n = d_k·7^k + ... + d_1·7 + d_0` với `0 ≤ d_i ≤ 6`, `d_k ≠ 0` (biểu diễn này **duy nhất**).
- `n % 7 = d_0` vì mọi số hạng khác chia hết cho 7 và `0 ≤ d_0 < 7`.
- `n // 7 = d_k·7^(k-1) + ... + d_1` — chính là số có các chữ số còn lại.

**Invariant:** sau `i` vòng, `digits` chứa đúng `d_0, ..., d_(i-1)` và `n` hiện tại có biểu diễn hệ 7 là `d_k...d_i`. Khi `n = 0`, ta đã lấy đủ các chữ số; đảo lại cho thứ tự từ hàng cao xuống thấp. Không có số 0 thừa ở đầu vì vòng dừng ngay khi phần còn lại bằng 0.

Số âm: biểu diễn của `-n` chính là `"-"` + biểu diễn của `n` (đề quy ước như vậy, ví dụ `-7 → "-10"`).

## 10. Độ phức tạp

Gọi `d` = số chữ số hệ 7 của `|num|` = `⌊log₇|num|⌋ + 1` (với `|num| ≤ 10^7`, `d ≤ 9`).

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Trừ dần lũy thừa | `O(d)` | `O(d)` | hai lượt, dễ sai biên |
| Chia liên tiếp + đảo | `O(d)` | `O(d)` | chuẩn, khuyên dùng |
| Ghép về bên trái | `O(d²)` lý thuyết | `O(d)` | mỗi lần ghép copy chuỗi; d nhỏ nên nhanh |
| Đệ quy | `O(d)` lời gọi (ghép chuỗi `O(d²)` lý thuyết) | `O(d)` stack | gọn nhất |
| Dựng số "trông như hệ 7" | `O(d)` | `O(1)` phụ | số nguyên có `d` chữ số thập phân |

Có thể nói gọn: **`O(log |num|)` thời gian và bộ nhớ**.

In [14]:
import math
for x in [6, 7, 48, 49, 10**7]:
    d = len(to_base7_divide(x))
    print(f"{x:>9}: {d} chữ số hệ 7 | floor(log7)+1 = {math.floor(math.log(x, 7) + 1e-12) + 1}")

        6: 1 chữ số hệ 7 | floor(log7)+1 = 1
        7: 2 chữ số hệ 7 | floor(log7)+1 = 2
       48: 2 chữ số hệ 7 | floor(log7)+1 = 2
       49: 3 chữ số hệ 7 | floor(log7)+1 = 3
 10000000: 9 chữ số hệ 7 | floor(log7)+1 = 9


## 11. Bộ test

Nhóm test: ví dụ đề, `0`, `±1`, ranh giới lũy thừa (`6, 7, 48, 49, 342, 343`), cực trị `±10^7`, và **đối chiếu 5 cách + `int(s, 7)` trên toàn bộ đoạn `[-3000, 3000]`**.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

s = Solution()
check("đề 1", s.convertToBase7(100), "202")
check("đề 2", s.convertToBase7(-7), "-10")
check("số 0", s.convertToBase7(0), "0")
check("1", s.convertToBase7(1), "1")
check("-1", s.convertToBase7(-1), "-1")
check("6 (chữ số lớn nhất)", s.convertToBase7(6), "6")
check("49 = 7^2", s.convertToBase7(49), "100")
check("48 = 7^2 - 1", s.convertToBase7(48), "66")
check("343 = 7^3", s.convertToBase7(343), "1000")
check("10^7", s.convertToBase7(10**7), "150666343")
check("-10^7", s.convertToBase7(-10**7), "-150666343")

PASS đề 1: '202'
PASS đề 2: '-10'
PASS số 0: '0'
PASS 1: '1'
PASS -1: '-1'
PASS 6 (chữ số lớn nhất): '6'
PASS 49 = 7^2: '100'
PASS 48 = 7^2 - 1: '66'
PASS 343 = 7^3: '1000'
PASS 10^7: '150666343'
PASS -10^7: '-150666343'


In [16]:
funcs = [to_base7_greedy, to_base7_divide, to_base7_prepend, to_base7_recursive, to_base7_numeric,
         Solution().convertToBase7, SolutionRecursive().convertToBase7]
for x in range(-3000, 3001):
    outs = {f(x) for f in funcs}
    assert len(outs) == 1, (x, outs)
    out = outs.pop()
    assert int(out, 7) == x, (x, out)
    assert out == "0" or not out.lstrip("-").startswith("0"), ("số 0 thừa ở đầu", x, out)
print("6001 giá trị trong [-3000, 3000]: 7 cài đặt khớp nhau và int(s, 7) đọc lại đúng — PASS")

6001 giá trị trong [-3000, 3000]: 7 cài đặt khớp nhau và int(s, 7) đọc lại đúng — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: quên trường hợp `0`
Vòng `while n > 0` không chạy → trả `""`.

### Lỗi 2: dùng `%` trực tiếp trên số âm
`-100 % 7 = 5` trong Python → ra chữ số sai.

### Lỗi 3: quên đảo chuỗi
Ra `"202"` đúng một cách "may mắn" vì là palindrome! Thử `-2024` mới thấy lỗi: ra `"-1265"` thay vì `"-5621"`.

### Lỗi 4: dùng `/` thay vì `//`
`100 / 7 = 14.285...` → vòng lặp chạy rất lâu với số thực và chữ số sai.

In [17]:
def buggy_no_zero(num):
    n, digits = abs(num), []
    while n > 0:
        digits.append(str(n % 7)); n //= 7
    return ("-" if num < 0 else "") + "".join(reversed(digits))

def buggy_negative_mod(num):
    if num == 0:
        return "0"
    n, digits = num, []
    for _ in range(12):               # giới hạn vòng để demo không chạy mãi
        if n == 0:
            break
        digits.append(str(n % 7)); n //= 7
    return "".join(reversed(digits)) + ("  (n vẫn chưa về 0!)" if n != 0 else "")

def buggy_no_reverse(num):
    if num == 0:
        return "0"
    n, digits = abs(num), []
    while n > 0:
        digits.append(str(n % 7)); n //= 7
    return ("-" if num < 0 else "") + "".join(digits)

print("quên 0     :", repr(buggy_no_zero(0)), "| đúng:", repr(to_base7_divide(0)))
print("% số âm    :", buggy_negative_mod(-100), "| đúng:", to_base7_divide(-100))
print("quên đảo   :", buggy_no_reverse(100), "(may mắn)", "|", buggy_no_reverse(-2024), "vs đúng", to_base7_divide(-2024))
print("dùng /     :", 100 / 7, "-> không phải số nguyên")

quên 0     : '' | đúng: '0'
% số âm    : 666666666465  (n vẫn chưa về 0!) | đúng: -202
quên đảo   : 202 (may mắn) | -1265 vs đúng -5621
dùng /     : 14.285714285714286 -> không phải số nguyên


Với `n` âm, `n //= 7` tiến về `-1` và **kẹt ở `-1`** mãi (`-1 // 7 = -1`) — vòng `while n != 0` sẽ chạy vô hạn. Đó là lý do bản demo phải giới hạn 12 vòng.

## 13. Mẹo nhận dạng pattern

- "Đổi sang hệ cơ số b", "biểu diễn nhị phân/hex", "đếm chữ số", "tổng chữ số" → **vòng `% b` / `// b`**.
- Chữ số ra **từ phải sang trái** → nhớ **đảo** (hoặc ghép về bên trái).
- Luôn hỏi: **0** thì sao? **số âm** thì sao?
- Để test: Python đọc chuỗi hệ b bằng `int(s, b)` (2 ≤ b ≤ 36); có sẵn `bin`, `oct`, `hex` cho hệ 2/8/16.

In [18]:
print(bin(10), oct(64), hex(255))
print(int("1010", 2), int("ff", 16), int("z", 36))

0b1010 0o100 0xff
10 255 35


## 14. Biến thể

### 14.1. Hệ cơ số bất kỳ 2..36 (chữ số > 9 dùng chữ cái)

In [19]:
DIGITS = "0123456789abcdefghijklmnopqrstuvwxyz"

def to_base(num, base):
    if num == 0:
        return "0"
    n, out = abs(num), []
    while n:
        n, r = divmod(n, base)
        out.append(DIGITS[r])
    return ("-" if num < 0 else "") + "".join(reversed(out))

print(to_base(255, 16), to_base(10, 2), to_base(100, 7), to_base(-35, 36))
assert all(int(to_base(x, b), b) == x for b in range(2, 37) for x in range(-200, 201))
print("to_base khớp int(s, b) với mọi b trong 2..36 — PASS")

ff 1010 202 -z
to_base khớp int(s, b) với mọi b trong 2..36 — PASS


### 14.2. Excel column title (LeetCode 168) — "hệ 26 không có chữ số 0"
`A..Z` ứng với `1..26`, nên mỗi bước phải `n -= 1` trước khi lấy dư. Đây là biến thể "lệch 1" kinh điển.

In [20]:
def excel_title(n):
    out = []
    while n:
        n -= 1
        n, r = divmod(n, 26)
        out.append(chr(ord("A") + r))
    return "".join(reversed(out))

print(excel_title(1), excel_title(26), excel_title(27), excel_title(701))   # A Z AA ZY

A Z AA ZY


## 15. Bài tập (đáp án ở cuối)

### Bài 1 — Tính tay
Đổi `343` và `-50` sang hệ 7 bằng bảng chia liên tiếp.

### Bài 2 — Điền TODO

In [21]:
def exercise_base7(num):
    if num == 0:
        return "0"
    # TODO 1: tách dấu, n = abs(num)
    # TODO 2: vòng while lấy n % 7, n //= 7
    # TODO 3: đảo, gắn dấu
    pass

# assert exercise_base7(100) == "202"
# assert exercise_base7(-7) == "-10"

### Bài 3 — Đọc chuỗi hệ 7 (không dùng `int(s, 7)`)
Viết `from_base7(s)` trả về số nguyên. Gợi ý: Horner.

In [22]:
def from_base7(s):
    # TODO
    pass

# assert from_base7("202") == 100
# assert from_base7("-10") == -7

### Bài 4 — Tổng chữ số hệ 7
Trả về tổng các chữ số của `|num|` trong hệ 7 **mà không tạo chuỗi**.

In [23]:
def digit_sum_base7(num):
    # TODO
    pass

# assert digit_sum_base7(100) == 4     # "202"
# assert digit_sum_base7(-2024) == 14  # "5621"

### Bài 5 — Đổi sang hệ âm (-2) (LeetCode 1017, mini challenge)
Chữ số chỉ là `0` hoặc `1`. Gợi ý: với cơ số âm, số dư có thể âm → chỉnh lại.

In [24]:
def to_base_neg2(n):
    # TODO
    pass

# assert to_base_neg2(2) == "110"
# assert to_base_neg2(3) == "111"
# assert to_base_neg2(0) == "0"

### Bài 6 — Số đối xứng trong hệ 7
Trả về list các số `x` trong `[1, limit]` mà biểu diễn hệ 7 là palindrome.

In [25]:
def base7_palindromes(limit):
    # TODO
    pass

# assert base7_palindromes(20) == [1, 2, 3, 4, 5, 6, 8, 16]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Làm như bảng trace mục 6 | 343 = 7³ | `-50`: làm với 50 rồi thêm dấu |
| 2 | Xem mục 5 | `digits.append(str(n % 7))` | `"-" + result if negative else result` |
| 3 | `"202"` = ((2)·7 + 0)·7 + 2 | `value = value * 7 + int(ch)` | xử lý dấu bằng `s[0] == "-"` |
| 4 | Không cần chuỗi | cộng `n % 7` | `total += n % 7; n //= 7` |
| 5 | `n = q·(-2) + r` với `r ∈ {0, 1}` | nếu `r < 0`: `r += 2`, `q += 1` | dùng `divmod(n, -2)` rồi chỉnh |
| 6 | Dùng lại `to_base7_divide` | `s == s[::-1]` | list comprehension trên `range(1, limit + 1)` |

## 17. Đáp án tham khảo

**Bài 1:** `343 = 7³ → "1000"`; `50 = 7·7 + 1 → 50 % 7 = 1, 7 % 7 = 0, 1 → "101"`, nên `-50 → "-101"`.

In [26]:
trace_base7(343)
print()
trace_base7(-50)

num = 343
bước 1: n=  343  n%7=0  n//7=  49  digits=['0']
bước 2: n=   49  n%7=0  n//7=   7  digits=['0', '0']
bước 3: n=    7  n%7=0  n//7=   1  digits=['0', '0', '0']
bước 4: n=    1  n%7=1  n//7=   0  digits=['0', '0', '0', '1']
đảo + dấu -> '1000' | kiểm tra int(.,7) = 343

num = -50
bước 1: n=   50  n%7=1  n//7=   7  digits=['1']
bước 2: n=    7  n%7=0  n//7=   1  digits=['1', '0']
bước 3: n=    1  n%7=1  n//7=   0  digits=['1', '0', '1']
đảo + dấu -> '-101' | kiểm tra int(.,7) = -50


'-101'

In [27]:
def answer_exercise_base7(num):
    if num == 0:
        return "0"
    negative, n, digits = num < 0, abs(num), []
    while n > 0:
        digits.append(str(n % 7))
        n //= 7
    result = "".join(reversed(digits))
    return "-" + result if negative else result

def answer_from_base7(s):
    sign = -1 if s[0] == "-" else 1
    value = 0
    for ch in s.lstrip("-"):
        value = value * 7 + int(ch)
    return sign * value

def answer_digit_sum_base7(num):
    n, total = abs(num), 0
    while n:
        total += n % 7
        n //= 7
    return total

def answer_to_base_neg2(n):
    if n == 0:
        return "0"
    out = []
    while n != 0:
        n, r = divmod(n, -2)
        if r < 0:          # Python: divmod với số chia âm cho r <= 0
            r += 2
            n += 1
        out.append(str(r))
    return "".join(reversed(out))

def answer_base7_palindromes(limit):
    res = []
    for x in range(1, limit + 1):
        s = to_base7_divide(x)
        if s == s[::-1]:
            res.append(x)
    return res

assert answer_exercise_base7(100) == "202"
assert answer_exercise_base7(-7) == "-10"
assert answer_from_base7("202") == 100 and answer_from_base7("-10") == -7
assert all(answer_from_base7(to_base7_divide(x)) == x for x in range(-500, 501))
assert answer_digit_sum_base7(100) == 4
assert answer_digit_sum_base7(-2024) == 14
assert answer_to_base_neg2(2) == "110" and answer_to_base_neg2(3) == "111" and answer_to_base_neg2(0) == "0"
for x in range(0, 200):   # kiểm tra lại hệ -2
    s = answer_to_base_neg2(x)
    assert sum(int(c) * (-2) ** i for i, c in enumerate(reversed(s))) == x
assert answer_base7_palindromes(20) == [1, 2, 3, 4, 5, 6, 8, 16]
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
def to_base(num, b):
    if num == 0:
        return "0"                      # 1. số 0
    n, out = abs(num), []               # 2. tách dấu
    while n:
        n, r = divmod(n, b)             # 3. lấy chữ số cuối, bỏ chữ số cuối
        out.append(str(r))
    s = "".join(reversed(out))          # 4. đảo
    return "-" + s if num < 0 else s    # 5. gắn dấu
```

Câu nhớ: **"Chia lấy dư, đọc ngược từ dưới lên."**

### Checklist
- [ ] `num == 0` → `"0"`.
- [ ] Làm việc với `abs(num)`, gắn `-` ở cuối.
- [ ] Dùng `//` (chia nguyên), không phải `/`.
- [ ] Đảo thứ tự chữ số (thử với số **không** đối xứng, ví dụ `-2024`).
- [ ] Trả về `str`, không phải `int`.
- [ ] Time/Space `O(log₇ |num|)`.